# Pyomo.DoE: Optimization

In the pseudo A-optimality workbook, we formulated model-based design of experiments as an optimal control problem using one objective function: pseudo A-optimality, based on $\log_{10}(\mathrm{trace}(\mathrm{FIM}))$.

In this workbook, we extend that idea to additional optimality criteria. We will use Pyomo.DoE to formulate, initialize, and solve experiment design problems using D-, A-, E-, and ME-optimality objectives. This lets us compare how the choice of design criterion changes the optimized input profile, $u(t)$, and the information content of the resulting experiment. Re-stating the optimal control formulation, we maximize a scalar-valued function $\psi(\cdot)$ of the Fisher information matrix, $\mathbf{M}$, subject to the TC Lab dynamic model, heater-input bounds, and initial conditions. 

Maximize a scalar-valued function $\psi(\cdot)$ of the Fisher information matrix $\mathbf{M}$:

$$
\begin{align*}
\max_{u} \quad & \psi(\mathbf{M}(u) + \mathbf{M}_0) \\
\mathrm{s.t.} \quad & C_p^H \frac{dT_H}{dt} = U_a (T_{amb} - T_H) + U_b (T_S - T_H) + \alpha P u(t)\\
& C_p^S \frac{dT_S}{dt} = U_b (T_H - T_S)  \\
& 0\% \leq u(t) \leq 100 \% \\
& T_H(t_0) = T_{amb} \\
& T_S(t_0) = T_{amb}
\end{align*}
$$

`Pyomo.DoE` automatically formulates, initializes, and solves this optimization problem for different choices of $\psi(\cdot)$, allowing us to compare D-, A-, E-, and ME-optimality criteria.

In [ ]:
import sys

# If running on Google Colab, install Pyomo and Ipopt via IDAES
on_colab = "google.colab" in sys.modules
if on_colab:
    !wget "https://raw.githubusercontent.com/dowlinglab/pyomo-doe/main/notebooks/tclab_pyomo.py"

# import TCLab model, simulation, and data analysis functions
from tclab_pyomo import (
    TC_Lab_data,
    TC_Lab_experiment,
    extract_results,
    extract_plot_results,
    results_summary,
)

# set default number of states in the TCLab model
number_tclab_states = 2

## Load experimental data (sine test)

We will load the sine test data to serve as an initial point. Recall our create model function will use supplied data to initialize the Pyomo model. Carefully initialization is often required for optimization of large-scale dynamic systems.

In [ ]:
import pandas as pd

if on_colab:
    file = "https://raw.githubusercontent.com/dowlinglab/pyomo-doe/main/data/tclab_sine_test_5min_period.csv"
else:
    file = '../data/tclab_sine_test_5min_period.csv'
df = pd.read_csv(file)
df.head()

We will create the data object for the design of experiment problems that follow

In [ ]:
# Here, we will induce a step size of 15 seconds, as to not give too many 
# degrees of freedom for experimental design.
skip =15

# Create the data object considering the new control points every 6 seconds
tc_data = TC_Lab_data(
    name="Sine Wave Test for Heater 1",
    time=df['Time'].values[::skip],
    T1=df['T1'].values[::skip],
    u1=df['Q1'].values[::skip],
    P1=200,
    TS1_data=None,
    T2=df['T2'].values[::skip],
    u2=df['Q2'].values[::skip],
    P2=200,
    TS2_data=None,
    Tamb=df['T1'].values[0],
)

## Calculate FIM at initial point (Values from $L_2$ regularization)

We will start by re-computing the prior FIM from $L_2$ regularization

In [ ]:
# Load Pyomo.DoE class
from pyomo.contrib.doe import DesignOfExperiments

from pyomo.environ import SolverFactory

# Copied from previous notebook
theta_values = {
    'Ua': 0.041705,
    'Ub': 0.012009,
    'inv_CpH': 0.167457,
    'inv_CpS': 4.545432,
}

In [ ]:
# Create experiment object for design of experiments
doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values, number_of_states=number_tclab_states)

# Create the design of experiments object using our experiment instance from above
TC_Lab_DoE = DesignOfExperiments(experiment=doe_experiment, 
                                 step=1e-2,
                                 scale_constant_value=1,
                                 scale_nominal_param_value=True, 
                                 tee=True,)

We will define a prior FIM from $L_2$ regularization 

In [ ]:

import numpy as np

cov = np.array([
    [1.857017e-10, -2.576198e-10, 1.402148e-09, -2.242347e-12],
    [-2.576198e-10, 1.624383e-07, 9.10987e-08, -6.32555e-05],
    [1.402148e-09, 9.10987e-08, 1.031454e-07, -3.890789e-05],
    [-2.242347e-12, -6.325555e-05, -3.890789e-05, 2.499914e-02],
])

FIM = np.linalg.inv(cov)

# Make sure resultant FIM is symmetric within numerical tolerance

FIM = 0.5*(FIM + FIM.T)


results_summary(FIM)


## Optimize next experiment (D-optimality)

We are now ready to solve the next experiment design problem. Here, we create a new `DesignOfExperiments` object and pass in the `prior_FIM`, which represents the information already collected from the previous experiment. By optimizing with this prior information included, Pyomo.DoE identifies the next best experiment to run according to the selected design criterion. In this section, we use D-optimality by setting `objective_option="determinant"`.

In [ ]:
# Create experiment object for design of experiments
doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values, number_of_states=number_tclab_states)

solver = SolverFactory("ipopt")
solver.options["max_iter"] = 3000
solver.options["tol"] = 1e-6
solver.options["linear_solver"] = "ma57"
solver.options["nlp_scaling_method"] = "gradient-based"
solver.options["acceptable_tol"] = 1e-4

# Create the design of experiments object using our experiment instance from above
TC_Lab_DoE_D = DesignOfExperiments(experiment=doe_experiment, 
                                 step=1e-2,
                                 scale_constant_value=1,
                                 scale_nominal_param_value=True,
                                 objective_option="determinant",  # Now we specify a type of objective, D-opt = "determinant"
                                 prior_FIM=FIM,  # We use the prior information from the existing experiment!,,,,,,,,,,,,,,,,,,,,,,,
                                 tee=True,
                                 solver = solver,)


TC_Lab_DoE_D.run_doe()

Now let's visualize the optimal experiment.

In [ ]:
dopt_pyomo_doe_results = extract_plot_results(None, TC_Lab_DoE_D.model.fd_scenario_blocks[0])

The D-optimal experiment produces an input profile that is close to a square wave, switching between full heater power and no heater power. This structure is intuitive: the heater stays on until the predicted sensor temperature reaches a high value, then turns off so the system can cool. The resulting temperature trajectory includes repeated heating and cooling cycles, which helps excite the model dynamics and improve parameter identifiability.

In this solution, the heater turns off when the predicted sensor temperature is near 85 °C, turns back on when the sensor cools to about 60 °C, and then allows the system to return toward 40 °C later in the experiment. Overall, the optimized design captures multiple heating and cooling events, giving the experiment richer dynamic information than the original sine-wave input.



We now analyze the predicted Fisher information matrix for the combined dataset: the experiment corresponding to the prior from $L_2$ regularization and this new optimized experiment.

First, let us check whether the combined FIM is full rank.

In [ ]:
import numpy as np, hashlib, json

def h(a):
    a = np.asarray(a, dtype=float)
    return hashlib.sha256(a.tobytes()).hexdigest()[:16]

print("theta_values:", theta_values)
print("theta hash:", h(list(theta_values.values()) if isinstance(theta_values, dict) else theta_values))
print("time len:", len(tc_data.time), "time[0],time[-1]:", float(tc_data.time[0]), float(tc_data.time[-1]))
print("u1 hash:", h(tc_data.u1), "T1 hash:", h(tc_data.T1), "T2 hash:", h(tc_data.T2))
print("prior_FIM hash:", h(FIM))
print("prior eig:", np.linalg.eigvalsh(0.5*(np.asarray(FIM)+np.asarray(FIM).T)))
print("solver options:", json.dumps(dict(solver.options), sort_keys=True))
print("DoE gradient method:", getattr(TC_Lab_DoE_D, "_gradient_method", None))
print("DoE fd_formula:", getattr(TC_Lab_DoE_D, "fd_formula", None))
print("DoE Cholesky_option:", getattr(TC_Lab_DoE_D, "Cholesky_option", None))
print("DoE use_grey_box:", getattr(TC_Lab_DoE_D, "use_grey_box", None))


In [ ]:
FIM_new = TC_Lab_DoE_D.results["FIM"]

FIM_array = FIM_new.to_numpy(dtype=float) if isinstance(FIM, pd.DataFrame) else np.asarray(FIM, dtype=float)

rank = np.linalg.matrix_rank(FIM_array)
num_params = FIM_array.shape[0]


print(f"FIM rank:{rank} out of {num_params} parameters")

The FIM has rank 4 out of 4, so it is not rank deficient. This means that the prior information and the new optimized experiment together provides independent information about all four parameter directions. Now let us examine the eigendecomposition

In [ ]:
results_summary(TC_Lab_DoE_D.results['FIM'])

The eigendecomposition shows that the least-informed parameter direction is still dominated by \(\mathrm{inv}\_C_p^S\), as indicated by the fourth eigenvector. However, the combined FIM is full rank, so this direction is no longer unidentifiable. These results suggest that the prior information and the new optimized experiment together provide enough independent information to estimate all four model parameters.

## Optimize next experiment (A-optimality)

Next, we repeat the same experiment design workflow using A-optimality instead of D-optimality. Instead of maximizing the determinant of the Fisher information matrix, A-optimality focuses on improving the average precision of the parameter estimates. We again include the prior FIM from the $L_2$ regularization, so the optimized input profile is chosen to add information to what we have already learned.

In [ ]:
# Create experiment object for design of experiments
doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values, number_of_states=number_tclab_states)
print(FIM)


# Create the design of experiments object using our experiment instance from above
TC_Lab_DoE_A = DesignOfExperiments(experiment=doe_experiment, 
                                 step=1e-2,
                                 scale_constant_value=1,
                                 scale_nominal_param_value=True,
                                 objective_option="trace",  # Now we specify a type of objective, A-opt = "trace"
                                 prior_FIM=FIM,  # We use the prior information from the same existing experiment as in the D-optimal case!
                                 tee=True,
                                 solver = solver,)

TC_Lab_DoE_A.run_doe()

In [ ]:
dopt_pyomo_doe_results = extract_plot_results(None, TC_Lab_DoE_A.model.fd_scenario_blocks[0])

Interestingly, the A-optimal design produces a different optimized experiment than the D-optimal design. The input still has a bang-bang, square-wave-like structure, switching between high and low heater power, but the timing of the switches is different. Compared with the D-optimal design, the predicted temperature trajectory remains in a higher temperature range for more of the experiment. This suggests that, for this example, the A-optimal criterion favors a different balance of heating and cooling behavior rather than simply reproducing the D-optimal input profile.

Next, let us examine the eigendecomposition of the FIM after this experiment

In [ ]:
results_summary(TC_Lab_DoE_A.results['FIM'])

Up to this point, we have explored D-optimality and A-optimality using algebraic formulations inside the Pyomo model. We now turn to eigenvalue-based criteria, E-optimality and modified E-optimality, which are more naturally handled using a GreyBox formulation.

Recall that the Fisher information matrix from the candidate experiment depends on the experiment design $u$:

$$
\mathbf{M}(u)
=
\mathbf{Q}(u)^\top \boldsymbol{\Sigma}_y^{-1}\mathbf{Q}(u),
$$

where $\mathbf{Q}(u)$ contains the sensitivities of the measured outputs with respect to the unknown parameters, and $\boldsymbol{\Sigma}_y$ is the measurement covariance matrix. When we include prior information from earlier experiments, the total information used in the design criterion is

$$
\mathbf{M}(u) + \mathbf{M}_0,
$$

where $\mathbf{M}_0$ is the prior Fisher information matrix.

D-optimality uses the determinant of $\mathbf{M}(u) + \mathbf{M}_0$, and A-optimality uses a trace-based measure. These objectives can be represented with algebraic formulations inside the Pyomo model.

E-optimality and modified E-optimality are different because they depend directly on the eigenvalues of $\mathbf{M}(u) + \mathbf{M}_0$:

$$
\text{E-optimality:} \qquad
\max_u \ \lambda_{\min}(\mathbf{M}(u) + \mathbf{M}_0),
$$

$$
\text{Modified E-optimality:} \qquad
\min_u \
\frac{\lambda_{\max}(\mathbf{M}(u) + \mathbf{M}_0)}
{\lambda_{\min}(\mathbf{M}(u) + \mathbf{M}_0)}.
$$

These criteria are useful when one parameter direction is much less informed than the others. In our TC Lab example, the eigendecomposition showed that the weakest-informed direction is dominated by $\mathrm{inv}\_C_p^S$, so E-optimality is a natural next criterion to test.

Because eigenvalue-based objectives are not as convenient to express using standard algebraic constraints, Pyomo.DoE uses a GreyBox formulation. The Pyomo model computes $\mathbf{M}(u) + \mathbf{M}_0$, and the GreyBox block evaluates the scalar criterion, such as the smallest eigenvalue or the condition-number-like ratio, while providing derivative information back to the optimizer.

## E-optimality using GreyBox Formulation

To handle eigenvalue-based design criteria, Pyomo.DoE provides a GreyBox objective formulation. This is useful for E-optimality because the design criterion depends on the smallest eigenvalue of the Fisher information matrix.

In the GreyBox formulation, the Pyomo model computes the Fisher information matrix, $\mathbf{M}(u)$, while the scalar design criterion is evaluated externally as

$$
\psi = \psi(\mathbf{M}(u)).
$$

For E-optimality, the GreyBox block evaluates

$$
\psi(\mathbf{M}(u)) = \lambda_{\min}(\mathbf{M}(u)).
$$

Thus:

- the Pyomo model computes $\mathbf{Q}(u)$ and $\mathbf{M}(u)$,
- the GreyBox block takes $\mathbf{M}(u)$ as input,
- the GreyBox block returns the smallest eigenvalue, $\lambda_{\min}(\mathbf{M}(u))$, as the scalar objective value.

The GreyBox callback also provides derivative information, such as

$$
\frac{\partial \psi}{\partial \mathbf{M}}
$$

and optionally second-derivative information. This allows a gradient-based optimizer such as IPOPT to solve the resulting problem efficiently.

For this example, E-optimality is useful because the earlier eigendecomposition showed one weakly informed parameter direction. By maximizing the smallest eigenvalue of the Fisher information matrix, E-optimality directly targets that least-informed direction.

We now formulate and solve the next experiment design problem using the GreyBox objective implementation with `objective_option="minimum_eigenvalue"`.

In [ ]:
# Create the design of experiments object for E-optimality

TC_Lab_DoE_E = DesignOfExperiments(
    experiment=doe_experiment,
    step=1e-2,
    scale_constant_value=1,
    scale_nominal_param_value=True,
    objective_option="minimum_eigenvalue",  # E-optimality
    prior_FIM=FIM,
    tee=True,
    solver = solver,
)

# Use the GreyBox objective implementation
TC_Lab_DoE_E.use_grey_box = True

# Solve the E-optimal design problem
TC_Lab_DoE_E.run_doe()


In [ ]:
print("=== E-optimal design summary ===")


results_summary(TC_Lab_DoE_E.results["FIM"])


The E-optimal design increases the smallest eigenvalue of the FIM from approximately $1.21 \times 10^4$ for the A-optimal design to approximately $1.39 \times 10^4$. This corresponds to an absolute increase of about $1.76 \times 10^3$, or roughly a 14.6% improvement in the least-informed parameter direction.

The eigendecomposition still shows that this weakest direction is dominated by $\mathrm{inv}\_C_p^S$. Thus, E-optimality does not change which parameter direction is least informed; instead, it directly increases the information available in that direction.

We next visualize the optimized experiment. 

In [ ]:
Eopt_pyomo_doe_results = extract_plot_results(None, TC_Lab_DoE_E.model.fd_scenario_blocks[0])

Recall that the D-optimal design increased the D-optimality criterion to about $30.32$, while the A-optimal design gave an A-optimality value of about $-4.08$. In contrast, the E-optimal design is judged by the smallest eigenvalue of the FIM, which increased to approximately $1.39 \times 10^4$.

The E-optimal solution again produces a bang-bang heater profile, but the switching times differ from the A- and D-optimal designs. For example, the E-optimal input turns off after the first heating interval at about 170 s, turns back on near 300 s, switches off again near 460 s, and begins the final heating interval near 600 s. These switching times are chosen to improve the weakest-informed parameter direction, rather than to optimize an average or determinant-based measure of information.

Specifically, here, the resulting experiment keeps the repeated heating and cooling structure, but adjusts the timing of the heating intervals to improve information in the least-informed parameter direction, which remains dominated by $\mathrm{inv}\_C_p^S$.

## Sensitivity Analysis

The E-optimal design increased the smallest eigenvalue of the FIM to approximately $1.39 \times 10^4$, improving the weakest-informed parameter direction. However, the eigendecomposition still shows that this direction is dominated by $\mathrm{inv}\_C_p^S$. The fourth eigenvector has coefficient $1.0000$ for $\mathrm{inv}\_C_p^S$, while the coefficients for $U_a$, $U_b$, and $\mathrm{inv}\_C_p^H$ are near zero. This indicates that the least-informed direction remains primarily associated with uncertainty in the sensor heat-capacity parameter.

We therefore perform a local sensitivity analysis around the nominal value $C_p^S = 0.22$ J/°C. This lets us check whether the E-optimal input profile and predicted temperature trajectory are robust to small changes in the parameter associated with the least-informed direction.

In [ ]:
import numpy as np

CpS_values = np.array([ 0.18, 0.19,0.2, 0.21, 0.22, 0.23, 0.24])
e_opt = np.zeros((len(CpS_values)))
u_solutions = np.zeros((len(CpS_values), len(tc_data.time)))
Ts_solutions = np.zeros((len(CpS_values), len(tc_data.time)))

for i, v in enumerate(CpS_values):

    print("\n********************\nCpS = ", v, " J/°C")

    theta_values_new = theta_values.copy()
    theta_values_new['inv_CpS'] = 1 / v

    # Create experiment object for design of experiments
    doe_experiment = TC_Lab_experiment(data=tc_data, theta_initial=theta_values_new, number_of_states=number_tclab_states)
    
    # Create the design of experiments object using our experiment instance from above
    TC_Lab_DoE = DesignOfExperiments(experiment=doe_experiment, 
                                     step=1e-2,
                                     scale_constant_value=1,
                                     scale_nominal_param_value=True, 
                                     tee=False,)

    FIM_new = TC_Lab_DoE.compute_FIM(method='sequential')

    # Create a new DoE object
    TC_Lab_DoE = DesignOfExperiments(experiment=doe_experiment, 
                                     step=1e-2,
                                     scale_constant_value=1,
                                     scale_nominal_param_value=True,
                                     objective_option="minimum_eigenvalue",  # We specify a type of objective, A-opt = "trace"
                                     prior_FIM=FIM,  # We use the prior information from the original experiment
                                     tee=False,
                                     solver = solver,)
    
   # Use the GreyBox objective implementation
    TC_Lab_DoE.use_grey_box = True
   
    TC_Lab_DoE.run_doe()

    
    pyomo_results = extract_results(TC_Lab_DoE.model.fd_scenario_blocks[0])

    results_summary(TC_Lab_DoE.results['FIM'])

    FIM_i = TC_Lab_DoE.results["FIM"]
    FIM_array = (
    FIM_i.to_numpy(dtype=float)
    if isinstance(FIM_i, pd.DataFrame)
    else np.asarray(FIM_i, dtype=float)
    )

    e_opt[i] = np.linalg.eigvalsh(FIM_array)[0]
    u_solutions[i, :] = pyomo_results.u1
    Ts_solutions[i, :] = pyomo_results.TS1_data

    print("********************\n")

Now let's visualize how the *E-optimality objective* changes as a function of $C_p^S$.

In [ ]:
import matplotlib.pyplot as plt


plt.plot(CpS_values, e_opt, marker='o')
plt.xlabel('$C_p^S$ ( J / °C )')
plt.ylabel(r'$\lambda_{\min}(M(u))$')
plt.show()

The E-optimality objective, $\lambda_{\min}(\mathbf{M}(u))$, increases over the tested local range of $C_p^S$. Across this range, the smallest eigenvalue increases from about $9.3 \times 10^3$ at $C_p^S = 0.18$ J/°C to about $1.39 \times 10^4$ at $C_p^S = 0.24$ J/°C.

This shows that the amount of information in the weakest-informed direction depends noticeably on the assumed value of $C_p^S$. Next, we check whether the optimized input profile and predicted temperature trajectory also change across this local range.

Next, we check whether the optimized input profile changes more noticeably than the scalar objective value.

In [ ]:
for i, v in enumerate(CpS_values):
    plt.plot(tc_data.time, u_solutions[i, :], label=f'$C_p^S$ = {v}')
plt.legend(ncol=2, loc='best')
plt.xlabel('Time (s)')
plt.ylabel('Heater Power (%)')
plt.show()

The corresponding optimal heater profiles are nearly identical across the tested values of $C_p^S$. All solutions retain the same bang-bang structure, switching between nearly full heater power and nearly zero heater power. Over this local range, changing $C_p^S$ has little effect on the E-optimal input design.

This suggests that the E-optimal design is locally robust to perturbations in the assumed sensor heat capacity.

Next, we examine the corresponding sensor-temperature profiles.

In [ ]:
for i, v in enumerate(CpS_values):
    plt.plot(tc_data.time, Ts_solutions[i, :], label=f'$C_p^S$ = {v}')
plt.legend(ncol=2, loc='best')
plt.xlabel('Time (s)')
plt.ylabel('Sensor Temperature (°C)')
plt.show()

The predicted sensor-temperature trajectories are also very similar across the tested values of $C_p^S$. The profiles follow the same heating and cooling pattern, with only small differences in peak temperature and cooling behavior.

Together, the heater profiles and temperature trajectories suggest that the E-optimal design is locally robust to small changes in $C_p^S$ around the nominal value, even though the value of the E-optimality objective itself changes across the tested range.

The local sensitivity analysis shows that the E-optimal design structure is fairly stable near the nominal value of $C_p^S$. The heater profiles and predicted temperature trajectories remain qualitatively similar across the tested range, even though the value of the E-optimality objective changes.

This tells us that E-optimality is doing what it is designed to do: it focuses on the weakest-informed direction by increasing the smallest eigenvalue of the FIM. However, to understand the overall quality of the information matrix, we should look at the entire eigenvalue spectrum, not only the smallest eigenvalue.

For the E-optimal design, the smallest eigenvalue is approximately

$$
\lambda_{\min} \approx 1.39 \times 10^4,
$$

while the largest eigenvalue is approximately

$$
\lambda_{\max} \approx 7.19 \times 10^9.
$$

Therefore, the ratio between the largest and smallest eigenvalues is approximately

$$
\frac{\lambda_{\max}}{\lambda_{\min}}
\approx
\frac{7.19 \times 10^9}{1.39 \times 10^4}
\approx 5.2 \times 10^5.
$$

This ratio is a measure of conditioning. A smaller ratio means the FIM has a more balanced distribution of information across parameter directions. A very large ratio means that some parameter directions are much better informed than others.

So, although E-optimality improves the least-informed direction, the FIM is still highly unevenly conditioned. This motivates modified E-optimality, which does not focus only on the smallest eigenvalue. Instead, it seeks to reduce the ratio between the largest and smallest eigenvalues, encouraging a more balanced information matrix.

## Modified E (ME)-optimality



Modified E-optimality considers the objective function:

$$
\frac{\lambda_{\max}(\mathbf{M}(u) + \mathbf{M}_0)}
{\lambda_{\min}(\mathbf{M}(u) + \mathbf{M}_0)}.
$$

This ratio measures the conditioning of the Fisher information matrix. If the ratio is large, then some parameter directions are much more informed than others. If the ratio is smaller, the information is more evenly distributed across parameter directions.

For this reason, ME-optimality seeks to minimize this ratio:

$$
\min_u
\frac{\lambda_{\max}(\mathbf{M}(u) + \mathbf{M}_0)}
{\lambda_{\min}(\mathbf{M}(u) + \mathbf{M}_0)}.
$$

ME-optimality asks whether we can design the next experiment to produce a better-conditioned information matrix, rather than focusing only on the weakest-informed direction. Let us look at the optimal experiment design

In [ ]:

# Create the design of experiments object
TC_Lab_DoE_ME = DesignOfExperiments(
    experiment=doe_experiment,
    step=1e-2,
    scale_constant_value=1,
    scale_nominal_param_value=True,
    objective_option="condition_number",  # ME-optimality
    prior_FIM=FIM,
    tee=True,
    solver = solver,
)

# Use the GreyBox objective implementation
TC_Lab_DoE_ME.use_grey_box = True

# Solve
TC_Lab_DoE_ME.run_doe()

In [ ]:
print("=== ME-optimal design summary ===")


results_summary(TC_Lab_DoE_ME.results["FIM"])

The ME-optimal design improves the conditioning of the FIM by reducing the spread between the largest and smallest eigenvalues. For this design, the smallest eigenvalue is approximately $1.67 \times 10^4$, while the largest eigenvalue is approximately $7.19 \times 10^9$. This gives a largest-to-smallest eigenvalue ratio of about $4.3 \times 10^5$.

Compared with the E-optimal design, where the smallest eigenvalue was approximately $1.39 \times 10^4$, modified E-optimality further increases the weakest-informed direction. The eigenvector associated with the smallest eigenvalue is still dominated by $\mathrm{inv}\_C_p^S$, so the least-informed direction has not changed. However, the information available in that direction has improved.

Thus, for this TC Lab example, ME-optimality gives a better-conditioned FIM while still improving the same weak parameter direction. Let us visualize the resulting optimal control trajectory.

In [ ]:
MEopt_pyomo_doe_results = extract_plot_results(None, TC_Lab_DoE_ME.model.fd_scenario_blocks[0])

The modified E-optimal input profile is noticeably different from the E-optimal design. Instead of using several shorter heating and cooling cycles, the ME-optimal design keeps the heater at full power for a long initial heating period, turns the heater off for an extended cooling period, and then returns to full power near the end of the experiment.

This behavior is consistent with the objective: ME-optimality is not only trying to increase the weakest-informed direction, but also to improve the conditioning of the FIM. The resulting trajectory explores a broader heating and cooling response, which helps balance information across parameter directions.

Now that we have solved the D-, A-, E-, and modified E-optimal design problems, we can compare the resulting Fisher information matrices using the same set of summary metrics. This comparison helps us see the tradeoffs between criteria: each design optimizes one objective, but may improve or degrade the others.

To compare the four optimized designs, we evaluate the same FIM summary metrics for each design. The table reports log-scaled values so the metrics can be compared on a similar numerical scale.

For the A-, D-, and E-style metrics, larger values indicate more information according to that criterion. For the condition-number metric, lower is better because it means the largest and smallest eigenvalues are closer together, corresponding to a better-conditioned FIM.

In [ ]:
def fim_condition_metrics(FIM):
    eigvals = np.linalg.eigvalsh(np.array(FIM, dtype=float))
    lambda_min = np.min(eigvals)
    lambda_max = np.max(eigvals)
    condition_number = lambda_max / lambda_min

    return {
        "λ_min": lambda_min,
        "λ_max": lambda_max,
        "κ(M) = λ_max / λ_min": condition_number,
    }


condition_df = pd.DataFrame(
    {
        "D-opt design": fim_condition_metrics(TC_Lab_DoE_D.results["FIM"]),
        "A-opt design": fim_condition_metrics(TC_Lab_DoE_A.results["FIM"]),
        "E-opt design": fim_condition_metrics(TC_Lab_DoE_E.results["FIM"]),
        "ME-opt design": fim_condition_metrics(TC_Lab_DoE_ME.results["FIM"]),
    }
).T

condition_df = condition_df.round(
    {
        "λ_min": 2,
        "λ_max": 2,
        "κ(M) = λ_max / λ_min": 2,
    }
)

print("=== FIM conditioning comparison ===")
display(condition_df)

The ME-optimal design gives the smallest condition number among the four designs. The condition number decreases from about $5.18 \times 10^5$ for the E-optimal design to about $4.29 \times 10^5$ for the ME-optimal design. This means the ME-optimal design produces a better-conditioned FIM.

The improvement comes mainly from increasing the smallest eigenvalue: $\lambda_{\min}$ increases from about $1.39 \times 10^4$ for the E-optimal design to about $1.67 \times 10^4$ for the ME-optimal design, while $\lambda_{\max}$ remains approximately $7.19 \times 10^9$ for all designs.


Overall, this example shows that the choice of design criterion changes what aspect of the Fisher information matrix is emphasized. D-optimality improves the determinant-based measure of total information, A-optimality targets an average uncertainty measure, E-optimality directly improves the weakest-informed parameter direction, and modified E-optimality improves the conditioning of the FIM.

For the TC Lab example, all four designs produce informative pulsed experiments, but they differ in how they distribute information across parameter directions. The weakest-informed direction remains associated with $\mathrm{inv}\_C_p^S$, which reflects uncertainty in the sensor heat-capacity parameter. E-optimality increases information in this direction, while modified E-optimality gives the best-conditioned FIM by reducing the spread between the largest and smallest eigenvalues.

The main takeaway is that there is no single universally “best” optimality criterion. The appropriate choice depends on the experimental goal: use D- or A-optimality when the goal is to improve overall information content, use E-optimality when the goal is to improve the least-informed direction, and use modified E-optimality when the goal is to obtain a more balanced, better-conditioned information matrix.

In [ ]:
import sys
import pyomo
import pyomo.version
import pyomo.contrib.doe

print("Python:", sys.executable)
print("Pyomo version:", pyomo.version.__version__)
print("Pyomo path:", list(pyomo.__path__))
print("Pyomo.DoE path:", pyomo.contrib.doe.__file__)

In [ ]:
import shutil
import sys
import pyomo
import pyomo.version
import pyomo.contrib.doe

print("Python:", sys.executable)
print("Pyomo version:", pyomo.version.__version__)
print("Pyomo.DoE path:", pyomo.contrib.doe.__file__)
print("Ipopt path:", shutil.which("ipopt"))